In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
import cv2
from tensorflow.keras.models import load_model
import numpy as np

In [ ]:
print(tf.config.list_physical_devices())

In [ ]:

model = load_model("best_inception_model.h5")

In [ ]:
labels={'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25, 'del': 26, 'nothing': 27, 'space': 28}
labels = {v: k for k, v in labels.items()}

In [ ]:
import cv2
import numpy as np
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
cap = cv2.VideoCapture(0)

IMG_SIZE = 224   # InceptionV3 input

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        h, w, _ = frame.shape

        # Get bounding box from hand landmarks
        x = [lm.x for lm in hand.landmark]
        y = [lm.y for lm in hand.landmark]

        x_min = int(min(x) * w)
        x_max = int(max(x) * w)
        y_min = int(min(y) * h)
        y_max = int(max(y) * h)

        # Expand box to look like dataset images
        expand = 40  # <-- very important!
        x_min = max(0, x_min - expand)
        y_min = max(0, y_min - expand)
        x_max = min(w, x_max + expand)
        y_max = min(h, y_max + expand)

        roi = frame[y_min:y_max, x_min:x_max]

        if roi.size > 0:
            # Make the ROI square with padding (MATCHES ASL dataset style)
            h2, w2 = roi.shape[:2]
            size = max(h2, w2)

            square = np.zeros((size, size, 3), dtype=np.uint8)
            y_offset = (size - h2) // 2
            x_offset = (size - w2) // 2
            square[y_offset:y_offset+h2, x_offset:x_offset+w2] = roi

            # Resize to Inception size
            img = cv2.resize(square, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype("float32") / 255.0
            img = np.expand_dims(img, axis=0)

            # Predict
            pred = model.predict(img, verbose=0)
            class_id = np.argmax(pred)
            conf = np.max(pred)

            label = f"{labels[class_id]} {conf:.2f}"
            cv2.putText(frame, label, (x_min, y_min - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0,255,0), 2)

    cv2.imshow("ASL Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()